# Python Types & Collections

*Variables · Mutability · Aliasing · is vs == · list/tuple/set/dict · collections · Real-World Production Scenarios*


---
## Introduction


# Fundamentals

*Run each cell with **Shift+Enter***

Python Foundations — Exhaustive Notebook
=========================================
Each section mirrors a Jupyter notebook cell:
  ① Rich explanation comment  (the "markdown" cell)
  ② Normal-behaviour demo with print output
  ③ Every GOTCHA demonstrated with BEFORE / AFTER evidence

Run:  python fundamentals.py


---
## 🧠 Notebook Mental Model: Python Types & Collections

> **Think of Python's type system as a spectrum from "fixed" to "flexible".**  
> Immutable objects (int, str, tuple) are like facts carved in stone — safe to share.  
> Mutable objects (list, dict, set) are like whiteboards — powerful but dangerous when shared carelessly.

### The Big Picture Map

```
                      PYTHON DATA MODEL
                      ─────────────────
  PRIMITIVE ──────── int  float  bool  str  bytes  None
                       │
  IMMUTABLE ─────────  │  (safe to share, hashable, can be dict keys)
                       │
  MUTABLE ──────────  list  dict  set  bytearray
                       │
  ALIASING TRAP ─────  ▲  Two names → ONE object → mutations are shared!
```

### Why / What / How / When at a Glance

| Topic | WHY it exists | WHAT to remember | HOW it works | WHEN to use |
|-------|--------------|-----------------|--------------|-------------|
| **`==` vs `is`** | Value equality ≠ identity | `is` checks same object in memory; `==` calls `__eq__` | CPython interns small ints/strings | Always use `==` for values; `is` only for `None`/singletons |
| **Mutability** | Enable in-place modification | Mutable = changeable after creation | Mutable objects have a fixed memory address; contents change | Use immutable for keys/hashing, mutable for accumulation |
| **Aliasing** | Memory efficiency | Two names can point to one object | Assignment copies the *reference*, not the object | Deep-copy when you need independence |
| **`list`** | Ordered, dynamic sequence | Backed by a resizable array; O(1) append | Overallocates capacity to amortize growth | Default ordered container |
| **`tuple`** | Fixed sequence + hashability | Immutable list → safe as dict key | Same layout as list but no resize | Return multiple values, dict keys, unpacking |
| **`set`** | Fast membership + deduplication | Hash table, O(1) average lookup | Stores only keys (no values) | `x in s`, deduplication, set algebra |
| **`dict`** | Key→value lookup | Ordered (3.7+), O(1) average | Open-addressing hash table | Any fast lookup by key |
| **`Counter`** | Frequency counting | dict subclass with `.most_common()` | Inherits dict; adds arithmetic | Word counts, histogram, top-N |
| **`defaultdict`** | Grouping without KeyError | Auto-creates missing keys | Calls factory on first access | Grouping, building adjacency lists |
| **`deque`** | O(1) both ends | Double-ended queue | Doubly-linked list of fixed blocks | BFS queues, sliding windows, logs |

### The Aliasing Mental Model (Most Common Bug)

```
  a = [1, 2, 3]   →  NAME "a"  ─→  [1, 2, 3] (object in memory)
  b = a            →  NAME "b"  ─╯
  b.append(4)      →  BOTH see [1, 2, 3, 4]  ← SURPRISE!

  Fix: b = a[:]        (shallow copy)
       b = a.copy()    (shallow copy)
       b = copy.deepcopy(a)  (deep copy, for nested)
```



In [ ]:
from __future__ import annotations

import copy, json, sys, tempfile, time, traceback
from collections import Counter, defaultdict, deque
from functools import reduce
from pathlib import Path
from enum import Enum


def sep(title: str) -> None:
    print(f"\n{'═'*64}\n  {title}\n{'═'*64}")

===========================================================================
1. VARIABLES, TYPES, `==` vs `is`
===========================================================================


### 🧠 Mental Model: Variables, Types & Identity (`==` vs `is`)

**WHY** — Python separates *value equality* (`==`) from *object identity* (`is`) because the same logical value can live in multiple memory locations, and sometimes you need to distinguish between them.

**WHAT** — Every Python object has three traits: **identity** (its address in memory, via `id()`), **type** (class), and **value**. Variable names are just labels pinned to objects.

**HOW** — `==` calls `__eq__` (compares values). `is` compares `id()` directly (same object?). CPython optimises by *interning* small integers (−5 to 256) and some strings, so `is` may accidentally return `True` for them — this is an implementation detail, not a language guarantee.

**WHEN** — Use `==` for values 99 % of the time. Use `is` only for:
- `x is None` (the canonical `None` check)
- `x is True` / `x is False` (very rare)
- Sentinel objects you deliberately created as identity markers

```
Memory diagram:

  list1 = [1,2,3]   →  id: 0xAB00  →  [1, 2, 3]
  list2 = [1,2,3]   →  id: 0xCC40  →  [1, 2, 3]
  list1 == list2    ✓  (same value)
  list1 is list2    ✗  (different objects)

  alias = list1     →  alias points to 0xAB00 (same!)
  alias is list1    ✓
```

**Gotcha** — Mutable aliasing: `alias = list1` does NOT copy; both names own the same list.


In [ ]:
def demo_types_and_identity() -> None:
    x = 42            # int (immutable)
    pi = 3.14         # float
    name = "Ada"      # str (immutable)
    ok = True         # bool
    nothing = None    # the "no value" singleton

    a, b = 1, 2
    a, b = b, a       # tuple unpacking -> swap with no temp
    assert (a, b) == (2, 1)

    # `==` compares VALUES; `is` compares IDENTITY (same object).
    list1 = [1, 2, 3]
    list2 = [1, 2, 3]
    assert list1 == list2       # equal values
    assert list1 is not list2   # different objects
    assert nothing is None      # `is` is the correct check for None

    # Mutable aliasing: two names, one object -> edits are shared.
    alias = list1
    alias.append(4)
    assert list1 == [1, 2, 3, 4]  # list1 changed through `alias`

    print("1. types/identity:", x, pi, name, ok, nothing, "| aliasing works")

===========================================================================
2. TRUTHINESS
===========================================================================


### 🧠 Mental Model: Truthiness

**WHY** — Python lets *any* object be used as a boolean condition so you can write `if items:` instead of `if len(items) > 0:`. This is the "pythonic" style.

**WHAT** — Python calls `bool(x)` before branching. An object is *falsy* if `bool(x) == False`; otherwise it's *truthy*.

**HOW** — Python checks in order:
1. Does `x` define `__bool__`? → use it.
2. Does `x` define `__len__`? → falsy if `len(x) == 0`.
3. Otherwise → truthy by default.

**Falsy values** (everything else is truthy):
```
0  0.0  0j  ""  b""  []  ()  {}  set()  None  False
```

**WHEN** — Use truthiness checks for empty containers / None guards:
```python
# Pythonic
if users:           # list not empty
if data is not None and data:  # None-safe then non-empty
if not error_msg:   # string is empty

# Un-pythonic (avoid)
if len(users) > 0:
if users != []:
```

**Gotcha** — `0` and `False` are falsy, so `if value:` fails for valid falsy inputs. Use `if value is not None:` when `0` is a valid value.


In [ ]:
def demo_truthiness() -> None:
    falsy = [0, 0.0, "", [], {}, set(), None, False]
    assert all(not bool(v) for v in falsy)
    # Pythonic: `if items:` means "if there are items".
    items: list[int] = []
    assert not items
    items = [1]
    assert items
    print("2. truthiness: empty containers/0/None are falsy")

===========================================================================
3. CONTROL FLOW
===========================================================================


### 🧠 Mental Model: Control Flow

**WHY** — Python's control flow is designed to be readable: the `for-else` construct, `match`, and `walrus` operator (`:=`) each solve a specific awkward pattern.

**WHAT / HOW / WHEN for each construct:**

| Construct | What | How | When |
|-----------|------|-----|------|
| `for-else` | `else` runs if the loop finished without `break` | Python tracks whether `break` was hit | Searching in a loop — replaces a `found` flag |
| `while-else` | Same as for-else | `else` fires when condition becomes False naturally | Retry loops with a "gave up" branch |
| `match` (3.10+) | Structural pattern matching | Compares shape and value of an object | Replace long `if/elif` chains on data structures |
| `:=` walrus | Assign + test in one expression | `(x := expr)` assigns then yields the value | Avoid re-computing in `while` and comprehension filters |

**Key Insight — `for-else` diagram:**
```
for item in collection:
    if condition(item):
        break         # ← else is SKIPPED when break fires
else:
    # only runs if NO break occurred (loop exhausted naturally)
    handle_not_found()
```

**Gotcha** — `for-else` with an empty iterable: the `else` branch *still* runs because there was no `break`.

**match bare name = capture variable** (common mistake):
```python
TARGET = 200
match code:
    case TARGET:  # ← WRONG: captures into a new var named TARGET
    case _ if code == TARGET:  # ← CORRECT: use a guard
```


In [ ]:
def classify(n: int) -> str:
    # match (Python 3.10+) — structural pattern matching replaces switch.
    match n:
        case 0:
            return "zero"
        case _ if n < 0:
            return "negative"
        case _:
            return "positive"


def first_prime_gap(limit: int) -> str:
    # for-else: the else runs only if the loop finished WITHOUT break.
    for n in range(2, limit):
        if all(n % d for d in range(2, int(n**0.5) + 1)):
            return f"first prime found: {n}"
    else:
        return "no prime found"


def demo_control_flow() -> None:
    assert classify(0) == "zero"
    assert classify(-5) == "negative"
    assert classify(9) == "positive"
    assert "prime" in first_prime_gap(20)
    print("3. control flow:", classify(-1), "|", first_prime_gap(20))

===========================================================================
4. THE FOUR BUILT-IN COLLECTIONS
===========================================================================


### 🧠 Mental Model: The Four Built-in Collections

**WHY** — Python ships four general-purpose containers because no single data structure is optimal for every use case. Choosing the right one is a key interview signal.

**Decision Tree:**
```
Need ordered data?
  ├─ Yes → will it change?
  │         ├─ Yes  → list  (dynamic array)
  │         └─ No   → tuple (immutable; hashable)
  └─ No  → need unique values?
            ├─ Yes  → set   (hash set, O(1) lookup)
            └─ No   → dict  (hash map, key→value, O(1) lookup)
```

**Complexity Cheat Sheet:**

| Operation | list | tuple | set | dict |
|-----------|------|-------|-----|------|
| Access by index | O(1) | O(1) | — | — |
| Lookup (`in`) | **O(n)** | **O(n)** | **O(1)** | **O(1)** |
| Insert at end | O(1)* | — | O(1)* | O(1)* |
| Insert at front | **O(n)** | — | O(1)* | — |
| Delete arbitrary | O(n) | — | O(1)* | O(1)* |

*amortised

**`collections` upgrades — use instead of reinventing:**

| Class | WHY | WHEN |
|-------|-----|------|
| `Counter` | Tallies elements; `.most_common(n)` in O(n log n) | Frequency, histogram, top-N |
| `defaultdict(list)` | No KeyError on first access | Grouping, adjacency lists, inverted index |
| `deque` | O(1) append/pop at **both** ends | BFS queue, sliding window, log ring buffer |
| `OrderedDict` | Remembers insertion order (pre-3.7 compat) | LRU cache implementations |
| `namedtuple` | Immutable record with field names | Lightweight data transfer objects |

**Mutability & Hashability:**
```
Hashable  →  can be a dict key / set member
Immutable →  always hashable (int, str, tuple, frozenset)
Mutable   →  never hashable (list, dict, set)
```


In [ ]:
def demo_collections() -> None:
    # list — ordered, mutable; comprehension is the Pythonic loop.
    squares = [n * n for n in range(5)]
    evens = [n for n in range(10) if n % 2 == 0]
    assert squares == [0, 1, 4, 9, 16]
    assert evens == [0, 2, 4, 6, 8]

    # tuple — immutable; hashable so it can be a dict key.
    point = (3, 4)
    lookup = {point: "origin-ish"}
    assert lookup[(3, 4)] == "origin-ish"

    # set — O(1) membership, dedupe, set algebra.
    a, b = {1, 2, 3}, {2, 3, 4}
    assert a & b == {2, 3}
    assert a | b == {1, 2, 3, 4}
    assert a - b == {1}
    assert 2 in a  # O(1)

    # dict — key -> value, O(1) lookup, safe access with .get.
    prices = {"apple": 3, "pear": 5}
    assert prices.get("plum", 0) == 0
    assert sorted(prices.items()) == [("apple", 3), ("pear", 5)]

    # collections upgrades
    counts = Counter("mississippi")
    assert counts["s"] == 4
    groups: dict[str, list[int]] = defaultdict(list)
    for n in range(6):
        groups["even" if n % 2 == 0 else "odd"].append(n)
    assert groups["even"] == [0, 2, 4]
    q: deque[int] = deque([1, 2, 3])
    q.appendleft(0)
    assert q.popleft() == 0

    print("4. collections: list/tuple/set/dict + Counter/defaultdict/deque")

===========================================================================
5. FUNCTIONS
===========================================================================


### 🧠 Mental Model: Functions

**WHY** — Functions encapsulate reusable logic, control scope, and are first-class objects in Python (can be stored in variables, passed as arguments, returned from other functions).

**Parameter kinds (strict left-to-right order):**
```
def fn(pos_only, /, normal, *args, kw_only, **kwargs):
         │            │        │        │         │
         │            │        │        │         └─ dict of extra keyword args
         │            │        │        └─────────── keyword-only args
         │            │        └──────────────────── extra positional args (tuple)
         │            └───────────────────────────── positional-or-keyword
         └────────────────────────────────────────── positional-only (before /)
```

**LEGB Scope Rule:**
```
  L → Local     (inside the current function)
  E → Enclosing (outer function's scope, for closures)
  G → Global    (module level)
  B → Built-in  (Python's built-in names: len, print, …)

Python searches L → E → G → B (first match wins)
```

**WHAT / HOW / WHEN:**

| Pattern | What | When |
|---------|------|------|
| Default `= None` | Guard against mutable defaults | Always use `None`, not `[]` / `{}` as default |
| `*args` | Collect extra positionals into a tuple | Variadic functions, forwarding |
| `**kwargs` | Collect extra keyword args into a dict | Config forwarding, decorator pass-through |
| Lambda | Anonymous single-expression function | `key=` argument in `sorted`/`min`/`max` |
| `functools.partial` | Pre-fill arguments | Currying, callback factories |
| `functools.reduce` | Fold a sequence into a single value | Running totals (but `sum` / loops are usually clearer) |

**Mutable Default Anti-Pattern:**
```python
# BAD — list is created once at def time, shared across all calls
def bad(items=[]):
    items.append(1); return items

# GOOD — None sentinel, fresh list each call
def good(items=None):
    items = items if items is not None else []
    items.append(1); return items
```


In [ ]:
def greet(name: str, greeting: str = "Hi", *args: str, **kwargs: object) -> str:
    extras = f" ({', '.join(args)})" if args else ""
    return f"{greeting}, {name}{extras}"


def add_item(x: int, items: list[int] | None = None) -> list[int]:
    # Correct: avoid the mutable-default trap by defaulting to None.
    items = items if items is not None else []
    items.append(x)
    return items


def demo_functions() -> None:
    assert greet("Ada") == "Hi, Ada"
    assert greet("Ada", "Hello", "eng", "math") == "Hello, Ada (eng, math)"

    # Mutable-default is NOT shared because we guard it.
    assert add_item(1) == [1]
    assert add_item(2) == [2]  # fresh list, no leakage

    double = lambda x: x * 2  # noqa: E731 (illustrative)
    assert double(21) == 42

    # Functional helpers (comprehensions are usually clearer).
    assert list(map(lambda x: x * 2, [1, 2, 3])) == [2, 4, 6]
    assert list(filter(lambda x: x > 1, [1, 2, 3])) == [2, 3]
    assert reduce(lambda acc, n: acc + n, [1, 2, 3, 4], 0) == 10

    print("5. functions: defaults, *args/**kwargs, lambda, map/filter/reduce")

===========================================================================
6. FILE HANDLING (context managers) + JSON
===========================================================================


### 🧠 Mental Model: File Handling & Context Managers

**WHY** — Files are operating-system resources: they must be *released* (closed) regardless of whether the code succeeds or raises an exception. Context managers automate this.

**WHAT** — `with open(...) as f:` is Python's resource management protocol. The `with` statement calls `__enter__` on entry and `__exit__` on exit — *even on exceptions*.

**HOW:**
```
with EXPR as VAR:      ┌──────────────────────────────────────────────┐
    BODY               │  1. EXPR.__enter__() → VAR                   │
                       │  2. BODY runs                                 │
                       │  3. EXPR.__exit__() always called             │
                       │     (exception info passed in if one occurred)│
                       └──────────────────────────────────────────────┘
```

**WHEN:**
- Always use `with open(...)` — never `f = open(); ... ; f.close()` (close is skipped on exceptions)
- Use `"r"` for text read, `"w"` for text write, `"rb"`/`"wb"` for binary
- Always specify `encoding="utf-8"` explicitly on text files

**JSON mental model:**
```
Python object  ──json.dumps()──►  JSON string  ──write to file──►  .json file
.json file  ──read from file──►  JSON string  ──json.loads()──►  Python object

Rule: json only handles: dict, list, str, int, float, bool, None
      datetime / custom objects need a custom encoder.
```


In [ ]:
def demo_files() -> None:
    with tempfile.TemporaryDirectory() as d:
        path = Path(d) / "data.txt"
        with open(path, "w", encoding="utf-8") as f:  # auto-closes on exit
            f.write("alpha\nbeta\ngamma\n")

        with open(path, encoding="utf-8") as f:
            lines = [line.strip() for line in f]  # lazy, memory-cheap iteration
        assert lines == ["alpha", "beta", "gamma"]

        cfg_path = Path(d) / "cfg.json"
        cfg_path.write_text(json.dumps({"env": "prod", "workers": 4}))
        cfg = json.loads(cfg_path.read_text())
        assert cfg["workers"] == 4

    print("6. files: with-statement auto-close, JSON round-trip")

===========================================================================
7. EXCEPTION HANDLING (EAFP)
===========================================================================


### 🧠 Mental Model: Exception Handling (EAFP)

**WHY** — Python favours *EAFP* (Easier to Ask Forgiveness than Permission) over *LBYL* (Look Before You Leap). Checking all conditions upfront is verbose and can have race conditions; just try and handle failures.

**WHAT** — The `try/except/else/finally` block is Python's structured error-handling construct. Custom exceptions communicate business errors in a domain vocabulary.

**HOW — anatomy of a try block:**
```
try:
    RISKY_CODE         ← runs first
except SomeError as e:
    HANDLE_ERROR       ← runs only if SomeError (or subclass) raised
except (A, B):
    HANDLE_MULTI       ← matches either A or B
else:
    SUCCESS_ONLY       ← runs only if NO exception was raised
finally:
    ALWAYS             ← runs unconditionally (even if re-raised!)
```

**WHEN — exception hierarchy decisions:**
```
BaseException
 └── Exception
      ├── ValueError    ← bad value for a valid type ("abc" as int)
      ├── TypeError     ← wrong type entirely
      ├── KeyError      ← missing dict key
      ├── IndexError    ← out-of-bounds list access
      ├── AttributeError← missing attribute
      ├── RuntimeError  ← generic runtime failure
      └── YourDomainError  ← subclass Exception for business errors
```

**Custom exception best practices:**
```python
class InsufficientFundsError(ValueError):  # inherits ValueError → catches both
    def __init__(self, balance, amount):
        super().__init__(f"Need {amount}, have {balance}")
        self.balance = balance   # carry structured data alongside message
        self.amount = amount
```

**EAFP vs LBYL:**
```python
# LBYL — verbose, possible TOCTOU race
if key in d:
    value = d[key]

# EAFP — pythonic
value = d.get(key, default)   # or:
try:
    value = d[key]
except KeyError:
    value = default
```


In [ ]:
class PaymentDeclined(Exception):
    """Domain-specific exception communicates a business error."""


def charge(balance: int, amount: int) -> int:
    if amount > balance:
        raise PaymentDeclined(f"need {amount}, have {balance}")
    return balance - amount


def safe_int(value: str, default: int = 0) -> int:
    # EAFP: try the operation, handle the specific failure.
    try:
        return int(value)
    except ValueError:
        return default


def demo_exceptions() -> None:
    assert safe_int("42") == 42
    assert safe_int("nope", default=-1) == -1

    try:
        charge(balance=100, amount=250)
    except PaymentDeclined as e:
        caught = str(e)
    else:  # pragma: no cover - illustrative
        caught = ""
    finally:
        cleanup = True  # `finally` always runs (cleanup)
    assert "need 250" in caught and cleanup
    print("7. exceptions: EAFP, custom exception, try/except/else/finally")


def main() -> None:
    print("=" * 68)
    print("PYTHON FOUNDATIONS — fundamentals.py")
    print("=" * 68)
    demo_types_and_identity()
    demo_truthiness()
    demo_control_flow()
    demo_collections()
    demo_functions()
    demo_files()
    demo_exceptions()
    print("-" * 68)
    print("All fundamentals demos passed ✔")

## ═══  EXHAUSTIVE GOTCHA NOTEBOOK  ══════════════════════════════════════════

In [ ]:
def notebook_section_1_names_and_identity() -> None:

## §1 · Names, Objects & Identity

In [ ]:
# A Python variable is a LABEL on an object, never a box.
    # Assigning b = a gives b the same label (same object).
    # Mutating through b is visible through a — same object, two names.

    a = [1, 2, 3]
    b = a                          # b is a second label on the SAME list
    b.append(4)
    print(f"a={a}  ← mutation through b is visible through a (same object)")
    assert a is b

    # Rebinding b does NOT affect a
    b = [99]
    print(f"a={a} b={b}  rebinding b has no effect on a")
    assert a == [1, 2, 3, 4]

    # id() returns the object's identity (CPython: memory address)
    x, y = 10, 10
    print(f"id(x)==id(y) for small int 10? {id(x)==id(y)}")   # True (cached)
    x2, y2 = 1000, 1000
    print(f"id(x2)==id(y2) for 1000?       {id(x2)==id(y2)}")  # may be False


def notebook_section_2_is_vs_eq() -> None:

## §2 · `is` vs `==` — The Small-Int/String Cache Trap

In [ ]:
# == tests VALUE (calls __eq__)
    # is  tests IDENTITY (same object)
    #
    # GOTCHA 1: CPython caches integers -5..256.
    #   `is` works for small ints by COINCIDENCE, not contract.
    # GOTCHA 2: String literals are interned at compile time.
    #   `is` often works on short string literals — also coincidental.
    # RULE: use `is` ONLY for None, True, False.

    a, b = 256, 256
    print(f"256 is 256 → {a is b}")   # True  (cached)
    c, d = 257, 257
    print(f"257 is 257 → {c is d}")   # False (outside cache, two objects)
    print(f"257 == 257 → {c == d}")   # True  (value equal — always use ==)

    s1 = "hello"; s2 = "hello"
    s3 = "hel" + "lo"
    print(f"literal 'hello' is 'hello' → {s1 is s2}")  # True (interned)
    print(f"'hel'+'lo'  is 'hello'     → {s3 is s1}")  # implementation-defined!

    # Correct: None check
    val = None
    assert val is None             # ✓
    assert not (val is not None)   # ✓
    # NEVER: if val == None (works but bad style; None has only one instance)


def notebook_section_3_truthiness() -> None:

## §3 · Truthiness — The Complete Picture

In [ ]:
# bool(x) calls: x.__bool__() → x.__len__() → True (default)
    # Falsy: None, False, 0, 0.0, 0j, "", b"", [], {}, set(), range(0)
    #
    # GOTCHA 1: A custom class with no __bool__/__len__ is ALWAYS truthy.
    # GOTCHA 2: __bool__ takes PRIORITY over __len__.
    # GOTCHA 3: `and`/`or` return the OBJECT, not just a bool.
    # GOTCHA 4: Short-circuit suppresses side-effects on the right operand.

    class AlwaysTruthy:        # no __bool__ or __len__ → truthy
        pass

    class LenZero:
        def __len__(self): return 0   # len=0 → falsy

    class BoolOverride:
        def __bool__(self): return True
        def __len__(self):  return 0  # __bool__ wins

    print(f"empty custom class: {bool(AlwaysTruthy())}")  # True
    print(f"__len__==0:         {bool(LenZero())}")       # False
    print(f"__bool__ overrides: {bool(BoolOverride())}")  # True  (not False!)

    # `or` / `and` return objects
    print(f"[] or 'default' → {[] or 'default'!r}")      # 'default'
    print(f"'a' or 'b'      → {'a' or 'b'!r}")           # 'a'
    print(f"0 and 1/0       → {0 and (1/0 if False else 'never')}")  # 0

    # Short-circuit: right side may not evaluate
    calls = []
    def side(v, label):
        calls.append(label); return v

    side(False, "A") and side(True, "B")   # B never runs
    print(f"A and B: only A ran → calls={calls}")   # ['A']

    calls.clear()
    side(True, "C") or side(True, "D")    # D never runs
    print(f"C or D: only C ran  → calls={calls}")   # ['C']


def notebook_section_4_operators() -> None:

## §4 · Operators — Every Edge Case

In [ ]:
# GOTCHA 1: // is FLOOR division (toward -∞), not truncation toward 0
    # GOTCHA 2: % sign matches the DIVISOR in Python
    # GOTCHA 3: ** is RIGHT-associative
    # GOTCHA 4: / always returns float; // returns int for int operands
    # GOTCHA 5: chained comparisons are mathematical, not C-style bit-ops

    print(f" 7 // 2  = { 7 // 2}")    #  3
    print(f"-7 // 2  = {-7 // 2}")    # -4  NOT -3  (floor, not truncate)
    print(f" 7 // -2 = { 7 // -2}")   # -4  NOT -3

    print(f" 7 % 3   = { 7 % 3}")     #  1
    print(f"-7 % 3   = {-7 % 3}")     #  2  NOT -1  (sign follows divisor 3)
    print(f" 7 % -3  = { 7 % -3}")    # -2  NOT  1  (sign follows divisor -3)

    print(f"2**3**2   = {2**3**2}")    # 512 = 2**(3**2) right-assoc, NOT (2**3)**2=64
    print(f"(2**3)**2 = {(2**3)**2}") # 64

    print(f"type(4/2)  = {type(4/2).__name__}")    # float
    print(f"type(4//2) = {type(4//2).__name__}")   # int

    # Chained comparison: 1 < x < 10 means (1<x) AND (x<10)
    x = 5
    print(f"1 < 5 < 10 → {1 < x < 10}")   # True  (mathematical)
    # In C: (1<5)>3  →  True>3  →  1>3  →  False — different semantics!
    print(f"C-style: (1<5)>3 = {(1<x)>3}")   # False

    # Augmented assignment on immutable: creates a NEW object
    n = 100; old_id = id(n)
    n += 1
    print(f"int n+=1 creates new object: {id(n) != old_id}")   # True

    # Bitwise on ints — useful for flags
    flags = 0b0000
    READ, WRITE, EXEC = 4, 2, 1
    flags |= READ | WRITE
    assert flags & READ  == READ   # test flag
    flags &= ~WRITE                # clear flag
    assert flags & WRITE == 0
    print(f"flag manipulation: flags after R+W then -W = {bin(flags)}")


def notebook_section_5_control_flow() -> None:

## §5 · Control Flow — for/while/match Nuances

In [ ]:
# GOTCHA 1: for-else — else runs when NO break occurred.
    #   NOT "if the loop ran 0 times" — it also runs on empty iterables!
    # GOTCHA 2: continue does NOT skip the else clause.
    # GOTCHA 3: match bare-name is a CAPTURE variable, not a comparison.
    # GOTCHA 4: match OR-pattern uses | not or.

    # for-else on empty iterable: else STILL runs
    found = False
    for n in []:   # empty — loop body never executes
        found = True; break
    else:
        print("for-else on empty: else STILL ran")   # prints!

    # for-else: break suppresses else
    for n in [1, 3, 4, 5]:
        if n % 2 == 0:
            print(f"found even {n}: else will NOT run"); break
    else:
        print("no even found: else ran")

    # GOTCHA: match bare name is a CAPTURE pattern.
    # Python raises a compile-time SyntaxError / SyntaxWarning if it detects
    # that a bare-name case shadows a variable and makes subsequent cases
    # unreachable. The correct way is to use literals, guards, or dotted names.
    STATUS_OK = 200

    # ✓ Correct: use a literal
    def match_literal(code):
        match code:
            case 200:   return "ok"
            case _:     return "other"

    # ✓ Correct: use a guard for runtime variable comparison
    def match_guard(code):
        match code:
            case c if c == STATUS_OK:   return f"ok (guard, code={c})"
            case _:                     return "other"

    # ✓ Correct: dotted name (class attribute / Enum value) is safe
    from enum import Enum
    class Status(Enum):
        OK = 200; NOT_FOUND = 404

    def match_enum(code):
        match code:
            case Status.OK:        return "ok (enum)"
            case Status.NOT_FOUND: return "not found (enum)"
            case _:                return "other"

    print(f"literal: {match_literal(200)!r}  {match_literal(404)!r}")
    print(f"guard:   {match_guard(200)!r}   {match_guard(404)!r}")
    print(f"enum:    {match_enum(Status.OK)!r}  {match_enum(Status.NOT_FOUND)!r}")

    # match OR-pattern: | (not `or`)
    def classify_char(c):
        match c:
            case 'a' | 'e' | 'i' | 'o' | 'u':   return "vowel"
            case _:                                return "other"

    print(f"'a' → {classify_char('a')}  'b' → {classify_char('b')}")


def notebook_section_6_list_gotchas() -> None:

## §6 · list Gotchas

In [ ]:
# GOTCHA 1: sort() is IN-PLACE and returns None.
    # GOTCHA 2: remove() removes only the FIRST occurrence.
    # GOTCHA 3: pop(0) is O(n); use deque.popleft() for O(1).
    # GOTCHA 4: Slice is a SHALLOW copy.
    # GOTCHA 5: Modifying list while iterating skips or double-visits items.
    # GOTCHA 6: [[0]*n]*m creates n×m with SHARED inner lists.

    # GOTCHA 1
    lst = [3, 1, 2]
    ret = lst.sort()
    print(f"lst.sort() returns: {ret!r}")  # None  — common bug: result = lst.sort()

    # GOTCHA 2
    lst2 = [1, 2, 3, 2, 4]
    lst2.remove(2)
    print(f"remove(2) removes only FIRST: {lst2}")   # [1,3,2,4]

    # GOTCHA 3 — timing pop(0) vs popleft
    big = list(range(5000))
    t0 = time.perf_counter()
    for _ in range(500): big.pop(0)
    t_list = time.perf_counter() - t0

    big2 = deque(range(5000))
    t0 = time.perf_counter()
    for _ in range(500): big2.popleft()
    t_deque = time.perf_counter() - t0

    print(f"list.pop(0)×500:    {t_list*1000:.2f}ms")
    print(f"deque.popleft()×500:{t_deque*1000:.2f}ms  (should be much faster)")

    # GOTCHA 4 — shallow slice
    nested = [[1, 2], [3, 4]]
    shallow = nested[:]
    shallow[0].append(99)
    print(f"original after shallow[0].append: {nested}")  # [[1,2,99],[3,4]]

    deep = copy.deepcopy([[1, 2], [3, 4]])
    deep[0].append(99)
    print(f"original after deep copy mutation: [[1, 2], [3, 4]] unchanged ✓")

    # GOTCHA 5 — modifying during iteration
    lst3 = [1, 2, 3, 4, 5, 6]
    for item in lst3:          # DON'T do this
        if item % 2 == 0:
            lst3.remove(item)  # shifts indices — skips items!
    print(f"buggy removal result: {lst3}")  # [1,3,5] but may miss items in other cases

    lst4 = [1, 2, 3, 4, 5, 6]
    lst4 = [x for x in lst4 if x % 2 != 0]   # ✓ correct: build new list
    print(f"comprehension filter: {lst4}")     # [1,3,5]

    # GOTCHA 6 — [[0]*3]*3 shares inner lists
    bad_matrix = [[0]*3]*3
    bad_matrix[0][0] = 99
    print(f"bad_matrix all rows changed: {bad_matrix}")   # [[99,0,0],[99,0,0],[99,0,0]]

    good_matrix = [[0]*3 for _ in range(3)]
    good_matrix[0][0] = 99
    print(f"good_matrix only row 0:     {good_matrix}")  # [[99,0,0],[0,0,0],[0,0,0]]


def notebook_section_7_dict_gotchas() -> None:

## §7 · dict Gotchas

In [ ]:
# GOTCHA 1: RuntimeError when modifying keys during iteration.
    # GOTCHA 2: setdefault INSERTS the key if missing (not just reads).
    # GOTCHA 3: Counter missing key returns 0 (not KeyError).
    # GOTCHA 4: defaultdict also inserts on access — can surprise you.
    # GOTCHA 5: d[k] raises KeyError; d.get(k) returns None/default.

    # GOTCHA 1
    d = {"a":1,"b":2,"c":3}
    try:
        for k in d:
            if k == "b": del d[k]
    except RuntimeError as e:
        print(f"Modify during iteration: {e}")
    # Fix: iterate over a snapshot
    d2 = {"a":1,"b":2,"c":3}
    for k in list(d2):
        if k == "b": del d2[k]
    print(f"Safe deletion result: {d2}")

    # GOTCHA 2 — setdefault inserts
    d3 = {}
    print(f"'x' in d3 before setdefault: {'x' in d3}")   # False
    d3.setdefault("x", [])
    print(f"'x' in d3 after setdefault:  {'x' in d3}")   # True (inserted!)

    # GOTCHA 4 — defaultdict inserts on access
    dd = defaultdict(list)
    _ = dd["missing_key"]          # creates the key!
    print(f"'missing_key' in dd: {'missing_key' in dd}")   # True (was inserted)

    # GOTCHA 5 — KeyError vs .get()
    d4 = {"x": 0}
    try:
        _ = d4["y"]
    except KeyError:
        print("d4['y'] raises KeyError")
    print(f"d4.get('y', 'NONE') = {d4.get('y', 'NONE')!r}")


def notebook_section_8_mutable_default_and_closure() -> None:

## §8 · Mutable Default + Closure Late-Binding

In [ ]:
# GOTCHA: Default args evaluated ONCE at definition.
    # GOTCHA: Closure captures the VARIABLE, not its value.

    def broken(item, acc=[]):
        acc.append(item); return acc

    print(broken(1))   # [1]
    print(broken(2))   # [1,2]  ← accumulates!
    print(broken(3))   # [1,2,3]

    def fixed(item, acc=None):
        if acc is None: acc = []
        acc.append(item); return acc

    print(fixed(1))    # [1]
    print(fixed(2))    # [2]  ✓ independent

    # Late-binding closure trap
    bad = [lambda: i for i in range(4)]
    print(f"late-binding: {[f() for f in bad]}")   # [3,3,3,3]

    good = [lambda i=i: i for i in range(4)]
    print(f"default-bind: {[f() for f in good]}")  # [0,1,2,3]


def notebook_section_9_exception_gotchas() -> None:

## §9 · Exception Handling Gotchas

In [ ]:
# GOTCHA 1: bare `except:` catches SystemExit & KeyboardInterrupt.
    # GOTCHA 2: finally's return OVERRIDES try's return.
    # GOTCHA 3: `as e` binding is DELETED after the except block ends.
    # GOTCHA 4: bare `raise` preserves traceback; `raise e` resets it.
    # GOTCHA 5: try/else runs ONLY when no exception was raised.

    # GOTCHA 2 — finally overrides try's return
    # NOTE: Python warns about `return` inside `finally`; the result variable
    # pattern below shows the same observable behaviour without the warning.
    result_box = []
    def finally_wins():
        try:
            result_box.append("try")
            return "try"
        finally:
            result_box.append("finally")
            # returning from finally overrides try's return:
            # return "finally"   ← do NOT do this in production
    ret = finally_wins()
    print(f"finally always runs: result_box={result_box}")
    print(f"return value (finally did NOT override here): {ret!r}")
    # To actually demonstrate the override we use exec to avoid compile warning:
    exec("""
def _fw():
    try: return 'from-try'
    finally: return 'from-finally'
print(f'finally return overrides try: {_fw()!r}')   # 'from-finally'
""")

    # GOTCHA 3
    try:
        raise ValueError("test")
    except ValueError as e:
        saved = str(e)
    try:
        print(e)                   # NameError — e was deleted
    except NameError:
        print(f"'e' deleted after except block (value was: {saved!r})")

    # GOTCHA 5 — else only on no-exception path
    def try_else(fail):
        log = []
        try:
            log.append("try")
            if fail: raise ValueError
        except ValueError:
            log.append("except")
        else:
            log.append("else")    # only if no exception
        finally:
            log.append("finally")
        return log

    print(f"no exception:  {try_else(False)}")   # try, else, finally
    print(f"with exception:{try_else(True)}")     # try, except, finally  (no else!)


def notebook_section_10_shallow_deep_copy() -> None:

## §10 · Shallow vs Deep Copy

In [ ]:
# Assignment (b=a):   NO copy; both names on same object
    # lst[:] / list(lst): shallow — new outer, shared inner objects
    # copy.copy():        shallow (calls __copy__)
    # copy.deepcopy():    deep — all nested objects are NEW

    nested = [[1, 2], [3, 4]]
    alias   = nested              # same object
    shallow = nested[:]           # new outer list, shared inner
    deep    = copy.deepcopy(nested)

    nested[0].append(99)
    print(f"alias sees mutation:   {alias}")    # [[1,2,99],[3,4]]
    print(f"shallow sees mutation: {shallow}")  # [[1,2,99],[3,4]]  ← shared inner!
    print(f"deep is unaffected:    {deep}")     # [[1,2],[3,4]]     ✓

    # Proof: inner objects of shallow are identical
    fresh = [[1,2],[3,4]]
    s = fresh[:]
    assert s[0] is fresh[0]       # same inner list object
    d = copy.deepcopy(fresh)
    assert d[0] is not fresh[0]   # fully independent inner list


def run_notebook() -> None:
    notebook_section_1_names_and_identity()
    notebook_section_2_is_vs_eq()
    notebook_section_3_truthiness()
    notebook_section_4_operators()
    notebook_section_5_control_flow()
    notebook_section_6_list_gotchas()
    notebook_section_7_dict_gotchas()
    notebook_section_8_mutable_default_and_closure()
    notebook_section_9_exception_gotchas()
    notebook_section_10_shallow_deep_copy()
    print("\n" + "═"*64)
    print("  NOTEBOOK COMPLETE — all gotchas demonstrated")
    print("═"*64)


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()
    run_notebook()

---
## Real-World Scenarios — BuildFast CI/CD Platform


# Python Core Scenarios

*Run each cell with **Shift+Enter***

Python Core — Real-World Scenarios
====================================
Every concept demonstrated through a CONCRETE PRODUCTION PROBLEM.

Scenario system: BuildFast — a SaaS CI/CD platform
  Users push code → BuildFast runs pipelines → returns results via API
  Language: Python · Framework: FastAPI · DB: PostgreSQL · Cache: Redis

Structure per topic:
  SCENARIO:    The production problem that hit the team
  WITHOUT:     What the code looked like and what broke
  WITH:        The fix and what it buys
  WHERE USED:  Real frameworks using this exact pattern
  GOTCHAS:     Edge cases that bit senior engineers

Run: python python_core_scenarios.py

In [ ]:
from __future__ import annotations
import copy, json, sys, time
from collections import Counter, defaultdict, deque
from functools import reduce

def sep(t): print(f"\n{'═'*64}\n  {t}\n{'═'*64}")
def h(t):   print(f"\n  ── {t} ──")

## SCENARIO 1 — MUTABLE DEFAULT ARGUMENTS

REAL INCIDENT at BuildFast

BuildFast stores pipeline run results. A senior engineer wrote a helper
to initialise pipeline metadata. In staging (small team, few runs) it
worked perfectly. In production, build metadata from User A's pipeline
appeared inside User B's pipeline results.

The bug: a mutable dict default argument. The same dict object was
reused across ALL calls. Every new pipeline inherited the accumulated
metadata from every previous pipeline in the process lifetime.

The fix required a hotfix deploy at 2AM. The incident affected 340 teams.

In [ ]:
def scenario_mutable_default() -> None:

## SCENARIO 1 · Mutable Default Argument — BuildFast Pipeline Incident

In [ ]:
h("WITHOUT — the bug that hit 340 teams")
    def init_pipeline_run_BAD(repo: str, metadata: dict = {}) -> dict:
        """Initialize a pipeline run record."""
        metadata["repo"] = repo
        metadata["started_at"] = time.time()
        return metadata   # ← returns the SHARED dict every time

    run1 = init_pipeline_run_BAD("org/frontend")
    run1["status"] = "building"
    run1["commit"] = "abc123"

    run2 = init_pipeline_run_BAD("org/backend")   # DIFFERENT repo
    print(f"Bug: run2 has run1's data:")
    print(f"  run2['commit'] = {run2.get('commit')!r}")   # 'abc123' from run1!
    print(f"  run2['status'] = {run2.get('status')!r}")   # 'building' from run1!
    print(f"  run1 is run2: {run1 is run2}")               # True — SAME object!

    h("WITH — sentinel pattern, always safe")
    def init_pipeline_run(repo: str, metadata: dict | None = None) -> dict:
        if metadata is None:
            metadata = {}      # fresh dict every call
        metadata["repo"]       = repo
        metadata["started_at"] = time.time()
        return metadata

    r1 = init_pipeline_run("org/frontend")
    r1["commit"] = "abc123"
    r2 = init_pipeline_run("org/backend")
    assert r1 is not r2
    assert "commit" not in r2           # r2 is clean
    print(f"\nFixed: r2 is isolated: 'commit' in r2 → {'commit' in r2}")
    print(f"Fixed: r1 is r2 → {r1 is r2}")

    h("WHERE THIS PATTERN IS USED IN FRAMEWORKS")
    print("""
  Django: Field(default=None) and using callable defaults
    class Build(Model):
        tags = JSONField(default=list)   ← list is a CALLABLE, called per instance
        # NOT: default=[]  ← that would be the mutable default trap!

  FastAPI: Request body models via Pydantic
    class PipelineRequest(BaseModel):
        env_vars: dict = {}         ← Pydantic copies per instance (safe)
        # But plain Python classes: use None + sentinel

  Dataclasses: field(default_factory=dict)
    @dataclass
    class PipelineConfig:
        env: dict = field(default_factory=dict)   ← safe
        # NOT: env: dict = {}                      ← shared across all instances!
""")

    h("GOTCHAS beyond the basics")
    # GOTCHA: class-level mutable attributes — the same trap at class scope
    class BadPipelineQueue:
        pending = []          # ONE list shared across ALL instances!

        def add(self, job: str) -> None:
            self.pending.append(job)   # mutates the CLASS attribute

    q1, q2 = BadPipelineQueue(), BadPipelineQueue()
    q1.add("build-frontend"); q2.add("build-backend")
    print(f"Class-level mutable: q1.pending = {q1.pending}")  # both jobs!
    print(f"Class-level mutable: q2.pending = {q2.pending}")  # same list!

    class GoodPipelineQueue:
        def __init__(self): self.pending = []  # instance attribute — isolated

    q3, q4 = GoodPipelineQueue(), GoodPipelineQueue()
    q3.add = lambda j: q3.pending.append(j)
    q4.add = lambda j: q4.pending.append(j)
    q3.add("build-frontend"); q4.add("build-backend")
    print(f"Instance attribute:  q3.pending = {q3.pending}")  # ['build-frontend']
    print(f"Instance attribute:  q4.pending = {q4.pending}")  # ['build-backend'] ✓

## SCENARIO 2 — ALIASING & OBJECT MODEL

REAL INCIDENT at BuildFast

BuildFast's API returns pipeline configs. A middleware layer adds
"request_id" to every config for tracing. After the change, some
pipeline configs in the database started having "request_id" fields —
data that should never persist.

The bug: the middleware was mutating the config dict it received
(which was the same object stored in the cache). The cache shared the
mutation. One request's tracing data permanently modified the cached
config.

In [ ]:
def scenario_aliasing() -> None:

## SCENARIO 2 · Aliasing — Request Tracing Mutation Incident

In [ ]:
h("WITHOUT — middleware mutates the cached config")
    pipeline_cache = {
        "pipe_001": {"name": "frontend-build", "steps": ["lint", "test", "build"]}
    }

    def add_tracing_bad(config: dict, request_id: str) -> dict:
        config["request_id"] = request_id   # MUTATES the dict in place!
        return config

    config = pipeline_cache["pipe_001"]        # gets a reference, not a copy
    traced = add_tracing_bad(config, "req_abc")
    print(f"Cache entry after tracing: {pipeline_cache['pipe_001']}")
    # request_id is now permanently in the cache!
    assert "request_id" in pipeline_cache["pipe_001"]   # LEAKED into cache

    h("WITH — copy before mutating")
    pipeline_cache2 = {
        "pipe_001": {"name": "frontend-build", "steps": ["lint", "test", "build"]}
    }

    def add_tracing_good(config: dict, request_id: str) -> dict:
        enriched = {**config, "request_id": request_id}  # new dict, original untouched
        return enriched

    config2 = pipeline_cache2["pipe_001"]
    traced2  = add_tracing_good(config2, "req_abc")
    assert "request_id" not in pipeline_cache2["pipe_001"]  # cache is clean!
    assert "request_id" in traced2
    print(f"Cache entry stays clean: 'request_id' in cache → {'request_id' in pipeline_cache2['pipe_001']}")
    print(f"Enriched response has it: 'request_id' in traced2 → {'request_id' in traced2} ✓")

    h("The identity rules — proof of the mechanism")
    a = [1, 2, 3]
    b = a             # alias — same object
    c = a[:]          # shallow copy — new list, same inner objects
    d = copy.deepcopy(a)  # deep copy — fully independent

    a[0] = 99
    print(f"\nb (alias) sees a[0]=99: {b[0]}")    # 99 — same object
    print(f"c (copy) is unaffected:  {c[0]}")     # 1 — independent
    print(f"d (deep)  is unaffected: {d[0]}")     # 1 — independent

    # Proof: shallow copy shares INNER mutable objects
    nested = [[1, 2], [3, 4]]
    shallow = nested[:]
    nested[0].append(99)
    print(f"\nShallow copy sees inner mutation: {shallow[0]}")  # [1,2,99]!
    deep = copy.deepcopy([[1, 2], [3, 4]])
    [[1, 2], [3, 4]][0].append(99)   # doesn't affect deep
    print(f"Deep copy is fully isolated:      {deep[0]}")       # [1,2]

    h("WHERE THIS IS SEEN IN FRAMEWORKS")
    print("""
  FastAPI: request.state is a mutable namespace — mutating it is safe
    because each request has its OWN state object.
    But if you cache a model object and mutate it: the cache is corrupted.

  SQLAlchemy: Session.expunge(obj) then modify — obj is detached from
    the session and your changes won't be persisted. The copy/detach
    distinction matters for exactly this aliasing reason.

  Redis caching: When you deserialise a cached value and mutate it,
    you're only mutating the local copy. To persist: re-serialise and
    call SET again. This is a feature of immutable cached values.

  Django ORM: Model.objects.get() returns the same model instance if
    the result is in the identity map. Mutating without .save() leaves
    in-memory state inconsistent with DB — classic aliasing trap.
""")

## SCENARIO 3 — COLLECTION CHOICE: O(n) vs O(1) IN PRODUCTION

REAL PERFORMANCE INCIDENT at BuildFast

BuildFast authenticates API calls against a list of valid API keys.
At launch: 100 customers, list works fine. At scale: 50,000 customers.
Each API call now checks `if key in api_keys_list` — O(50,000) per request.
At 1,000 req/sec: 50 MILLION comparisons per second just for auth.
P99 latency went from 8ms to 340ms. Root cause: wrong data structure.

In [ ]:
def scenario_collection_choice() -> None:

## SCENARIO 3 · Collection Choice — O(n) vs O(1) Auth at BuildFast

In [ ]:
import random, string, time

    def make_keys(n: int) -> list[str]:
        return ["bfk_" + "".join(random.choices(string.ascii_lowercase, k=16))
                for _ in range(n)]

    valid_keys = make_keys(50_000)
    test_key   = valid_keys[25_000]  # a key in the middle
    miss_key   = "bfk_doesnotexist_12345"

    h("WITHOUT — list membership check: O(n)")
    t0 = time.perf_counter()
    for _ in range(1000):
        _ = test_key in valid_keys        # O(n) linear scan each time!
    t_list_hit = time.perf_counter() - t0

    t0 = time.perf_counter()
    for _ in range(1000):
        _ = miss_key in valid_keys        # O(n) scans the WHOLE list every time
    t_list_miss = time.perf_counter() - t0

    print(f"list (50k keys) × 1000 hits:  {t_list_hit*1000:.1f}ms")
    print(f"list (50k keys) × 1000 misses:{t_list_miss*1000:.1f}ms  ← worst case")

    h("WITH — set/frozenset membership: O(1) average")
    valid_set   = frozenset(valid_keys)    # build once at startup, O(n)
    valid_dict  = {k: True for k in valid_keys}  # dict for O(1) with metadata

    t0 = time.perf_counter()
    for _ in range(1000):
        _ = test_key in valid_set         # O(1) hash lookup
    t_set_hit = time.perf_counter() - t0

    t0 = time.perf_counter()
    for _ in range(1000):
        _ = miss_key in valid_set         # O(1) even on miss!
    t_set_miss = time.perf_counter() - t0

    speedup = t_list_miss / max(t_set_miss, 0.00001)
    print(f"set  (50k keys) × 1000 hits:  {t_set_hit*1000:.2f}ms")
    print(f"set  (50k keys) × 1000 misses:{t_set_miss*1000:.2f}ms")
    print(f"\n  ⚡ Speedup: ~{speedup:.0f}× faster (set vs list)")
    print(f"  At 1000 req/sec this saves ~{(t_list_miss-t_set_miss)*1000:.0f}ms of CPU per second")

    h("deque vs list for task queues (O(1) vs O(n) left operations)")
    QUEUE_SIZE = 10_000

    lst = list(range(QUEUE_SIZE))
    t0 = time.perf_counter()
    for _ in range(500): lst.pop(0)       # O(n) each — shifts everything left
    t_list_pop = time.perf_counter() - t0

    dq = deque(range(QUEUE_SIZE))
    t0 = time.perf_counter()
    for _ in range(500): dq.popleft()     # O(1) each
    t_deque_pop = time.perf_counter() - t0

    print(f"\nTask queue popleft × 500:")
    print(f"  list.pop(0):    {t_list_pop*1000:.2f}ms  (O(n) = O({QUEUE_SIZE}))")
    print(f"  deque.popleft: {t_deque_pop*1000:.2f}ms  (O(1))")

    h("COLLECTION DECISION GUIDE — BuildFast's internal style guide")
    print("""
  Need O(1) membership test?          → set / frozenset / dict key
    (API key auth, feature flags, seen-IDs in dedup)

  Need ordered sequence + random access?  → list
    (pipeline steps, build log lines)

  Need O(1) append/pop BOTH ends?      → collections.deque
    (task queue, sliding window, BFS frontier)

  Need frequency count?                → collections.Counter
    (error type aggregation, build status summary)

  Need default value on missing key?   → collections.defaultdict
    (grouping builds by repo, adjacency list for deps)

  Need stable k→v with insertion order? → dict (Python 3.7+)
    (config, JSON payloads, response bodies)

  WHERE IN FRAMEWORKS:
  • FastAPI: response_model caches Schema objects in a dict for O(1) lookup
  • Django: User permission cache is a set for O(1) has_perm() check
  • Celery: task registry is a dict keyed by task name
  • Redis: SADD/SISMEMBER — set membership at distributed scale
""")

## SCENARIO 4 — EXCEPTION HANDLING: EAFP vs LBYL IN PRODUCTION APIs

REAL INCIDENT at BuildFast

BuildFast's webhook handler processed GitHub events. A LBYL-style
check for "is the repo key present?" had a TOCTOU race condition
in the async handler — the check passed on the coroutine's check,
but by the time the data was accessed another coroutine had modified
the dict. Result: KeyError crashing the webhook handler at peak load.

Separately: a bare `except:` block was silently swallowing payment
failures and returning HTTP 200. Builds appeared to start but never
actually ran. Users thought BuildFast was broken; it was silent failure.

In [ ]:
def scenario_exceptions() -> None:

## SCENARIO 4 · Exception Handling — Webhook & Payment Incidents

In [ ]:
h("LBYL race condition — the webhook incident")
    event_cache: dict = {}

    def handle_webhook_LBYL(event_id: str) -> dict:
        # Simulated async race: check passes, but data disappears between
        # check and access (another coroutine processed and deleted it)
        if "repo" in event_cache:         # LBYL check
            # --- another coroutine runs here, deletes event_cache["repo"] ---
            return {"repo": event_cache["repo"]}  # KeyError if it was deleted!
        return {"error": "no repo"}

    def handle_webhook_EAFP(event_id: str) -> dict:
        try:
            return {"repo": event_cache["repo"]}  # atomic: no race between check+access
        except KeyError:
            return {"error": "no repo"}

    # Demonstrate: LBYL is two operations, EAFP is one
    print("EAFP is race-condition-proof — one operation, not check+access")
    print(f"  EAFP result: {handle_webhook_EAFP('evt_001')}")

    h("Bare except — the payment silent-failure incident")
    def process_payment_BAD(amount: float, token: str) -> dict:
        try:
            if not token.startswith("tok_"):
                raise ValueError("invalid token format")
            # Simulate payment processor
            if amount > 10000: raise ConnectionError("payment gateway timeout")
            return {"status": "success", "charged": amount}
        except:                    # CATCHES EVERYTHING including SystemExit!
            return {"status": "success"}   # ← lies! payment failed but returns 200!

    # Builds appeared to start but billing never happened
    result = process_payment_BAD(99999, "tok_invalid_xxxxxxx")
    print(f"\nBare except lies: {result}")  # {"status": "success"} but payment failed!

    def process_payment_GOOD(amount: float, token: str) -> dict:
        try:
            if not token.startswith("tok_"):
                raise ValueError(f"Invalid token format: {token[:10]}...")
            if amount > 10000: raise ConnectionError("payment gateway timeout")
            return {"status": "success", "charged": amount}
        except ValueError as e:
            # Expected validation failure — user error, log as warning
            return {"status": "error", "code": "INVALID_TOKEN", "detail": str(e)}
        except ConnectionError as e:
            # Infrastructure failure — log as error, alert on-call
            raise RuntimeError("Payment processor unreachable") from e
        # Any OTHER exception propagates — we want to know about unexpected failures!

    try:
        process_payment_GOOD(99999, "tok_test_123")
    except RuntimeError as e:
        print(f"Gateway error propagated correctly: {e}")

    result2 = process_payment_GOOD(99.99, "not_a_tok")
    print(f"Invalid token handled: {result2}")

    h("try/except/else/finally — the complete pattern")
    def fetch_pipeline_config(pipeline_id: str, cache: dict, db: dict) -> dict:
        log = []
        try:
            log.append("try")
            config = cache[pipeline_id]    # check cache first
        except KeyError:
            log.append("except-cache-miss")
            config = db.get(pipeline_id)
            if config is None:
                raise ValueError(f"Pipeline {pipeline_id!r} not found") from None
            cache[pipeline_id] = config    # warm the cache on miss
        else:
            log.append("else-cache-hit")   # only if NO exception (cache hit)
        finally:
            log.append("finally-always")   # ALWAYS runs — audit log

        return {"config": config, "log": log}

    cache = {}; db = {"pipe_001": {"steps": ["test", "build"]}}
    r1 = fetch_pipeline_config("pipe_001", cache, db)
    r2 = fetch_pipeline_config("pipe_001", cache, db)  # cache hit now
    print(f"\nCache miss: {r1['log']}")   # try, except, finally
    print(f"Cache hit:  {r2['log']}")    # try, else, finally

    h("WHERE EXCEPTION PATTERNS ARE USED IN REAL FRAMEWORKS")
    print("""
  Django ORM — EAFP is the standard:
    try:
        user = User.objects.get(email=email)
    except User.DoesNotExist:           ← specific, not bare except
        return None

  FastAPI — HTTPException for user errors, let others propagate:
    @app.get("/pipelines/{id}")
    async def get_pipeline(id: str):
        try:
            return await db.fetch(id)
        except NotFound:
            raise HTTPException(404, "Pipeline not found")
        # Other DB errors propagate as 500 — correct behaviour

  SQLAlchemy — context manager guarantees rollback:
    with Session() as session:
        try:
            session.add(pipeline)
            session.commit()
        except IntegrityError:
            session.rollback()
            raise

  Python stdlib — contextlib.suppress for "don't care" errors:
    with suppress(FileNotFoundError):
        Path("/tmp/stale_lock").unlink()  ← no try/except needed
""")

## SCENARIO 5 — COMPREHENSIONS & GENERATORS IN ETL PIPELINES

REAL SCENARIO at BuildFast

BuildFast exports build telemetry for customers to analyze. Some
enterprise customers have 10M+ build events. Exporting them to CSV
with a naive list approach ran out of memory (16GB RAM) on the
export worker. The generator rewrite reduced peak memory from 14GB to ~8MB.

In [ ]:
def scenario_comprehensions_generators() -> None:

## SCENARIO 5 · Generators — 10M-Row ETL Export at BuildFast

In [ ]:
h("WITHOUT generators — entire dataset in RAM")
    def export_builds_bad(build_ids: range) -> list[dict]:
        results = []
        for bid in build_ids:
            # In reality: a DB row — each might be 1KB
            results.append({
                "id": bid,
                "repo": f"org/repo_{bid % 100}",
                "status": "success" if bid % 7 != 0 else "failed",
                "duration_ms": 12000 + (bid * 7 % 8000),
            })
        return results  # 10M dicts in RAM — 14GB!

    # Demonstrate with small numbers
    import sys
    small_list = export_builds_bad(range(10_000))
    list_size  = sys.getsizeof(small_list) + sum(sys.getsizeof(d) for d in small_list[:100]) * 100
    print(f"list of 10k rows: ~{list_size // 1024}KB in memory (for 10M → ~14GB)")

    h("WITH generator — O(1) memory, streams to disk")
    def export_builds_stream(build_ids: range):
        """Generator: yields one row at a time. Memory stays at ~one row."""
        for bid in build_ids:
            yield {
                "id": bid,
                "repo": f"org/repo_{bid % 100}",
                "status": "success" if bid % 7 != 0 else "failed",
                "duration_ms": 12000 + (bid * 7 % 8000),
            }

    def to_csv_line(row: dict) -> str:
        return ",".join(str(v) for v in row.values())

    # Entire pipeline: generator → filter → transform → write line by line
    # Peak memory: ONE row at a time regardless of total rows
    lines_written = 0
    for row in export_builds_stream(range(10_000)):
        line = to_csv_line(row)
        # f.write(line + "\n")  ← in real code: streamed to S3/disk
        lines_written += 1

    gen_size = sys.getsizeof(export_builds_stream(range(1)))
    print(f"Generator object size: {gen_size} bytes (regardless of 10M or 10B rows!)")
    print(f"Lines processed: {lines_written:,} — peak memory: ONE row ≈ ~300 bytes")

    h("Lazy pipeline — comprehension chain with no intermediate lists")
    # Realistic ETL: filter failed builds, compute P95 duration, top 5 repos
    all_builds = export_builds_stream(range(100_000))

    # Each step is a lazy generator — no intermediate materialisation
    failed     = (b for b in all_builds if b["status"] == "failed")
    long_fails = (b for b in failed if b["duration_ms"] > 15_000)
    repo_names = (b["repo"] for b in long_fails)

    # Counter only materialises the final aggregation
    from collections import Counter
    top_repos  = Counter(repo_names).most_common(5)
    print(f"Top repos with long failed builds: {top_repos}")
    print(f"Pipeline memory: O(1) — no intermediate lists stored")

    h("Comprehension vs loop — when each is right")
    builds = [
        {"id": i, "duration": 100 * i, "status": "success" if i % 3 else "failed"}
        for i in range(1, 11)
    ]

    # List comprehension: build result list (concise, readable)
    success_ids = [b["id"] for b in builds if b["status"] == "success"]
    print(f"\nList comp (materialise): {success_ids}")

    # Generator expression: feed into sum/max/Counter (lazy, no temp list)
    avg_duration = sum(b["duration"] for b in builds) / len(builds)
    print(f"Generator expr (lazy aggregate): avg_duration = {avg_duration:.0f}ms")

    # Dict comprehension: fast lookup table from a list
    id_to_build = {b["id"]: b for b in builds}
    print(f"Dict comp (lookup table): id_to_build[5] = {id_to_build[5]}")

    h("WHERE IN FRAMEWORKS")
    print("""
  Django QuerySets are LAZY generators:
    builds = Build.objects.filter(status="failed")  ← no DB query yet
    for b in builds: ...    ← DB query fires here, one page at a time
    # .iterator() streams rows without caching them in memory

  FastAPI streaming responses:
    @app.get("/export/builds")
    async def export():
        async def generate():
            async for row in db.stream_all_builds():
                yield row_to_csv(row) + "\\n"
        return StreamingResponse(generate(), media_type="text/csv")

  SQLAlchemy: .yield_per(1000) fetches 1000 rows at a time, not all:
    for build in session.execute(query).yield_per(1000):
        process(build)

  Celery: chord/chain composes tasks lazily — tasks as generators of work
""")


def main() -> None:
    print("="*70)
    print("PYTHON CORE — Real-World Scenarios (BuildFast CI/CD Platform)")
    print("="*70)
    scenario_mutable_default()
    scenario_aliasing()
    scenario_collection_choice()
    scenario_exceptions()
    scenario_comprehensions_generators()
    print("\n" + "="*70)
    print("Python core scenarios complete ✔")
    print("System: BuildFast — SaaS CI/CD serving 50k teams")


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()

---
## 🏆 Interview Questions — Python Foundations — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# Python Foundations — Interview Questions

> Format: 5 architectural deep-dive questions with answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. Why does `def f(x, cache={})` almost always misbehave, and what is actually happening?


**Deep dive.** A default argument is evaluated **once**, at function-definition
time, and that single object is reused on every call that doesn't pass the
argument. So a mutable default (`{}`, `[]`) becomes shared state that accumulates
across calls — the "cache" from one caller silently leaks into the next, and the
bug is invisible until a second call sees the first call's data. The mechanism is
the object/reference model: the default is stored on the function object
(`f.__defaults__`) and rebound to the parameter name each call, never re-created.
The fix is the sentinel idiom: default to `None` and build the fresh object
inside the body (`if cache is None: cache = {}`). The senior point is that this
isn't a quirk to memorize — it's the same mutability-plus-shared-reference fact
that causes aliasing bugs, which is why immutability at boundaries is a design
default, not a preference.

---

### Q2. Is Python pass-by-value or pass-by-reference, and what are the consequences?


**Deep dive.** Neither label fits; Python is **call-by-object-reference** (a.k.a.
call-by-sharing). The function receives a *reference* to the same object the
caller holds, but the parameter name is a new binding. Consequences follow from
mutability: if you **mutate** the argument in place (`lst.append(x)`,
`d[k] = v`), the caller sees it — the object is shared. If you **rebind** the
parameter (`lst = [...]`), the caller sees nothing — you only moved the local
name. So passing a mutable object is an implicit contract about who may mutate
it; passing an immutable one (int, str, tuple, frozen dataclass) is inherently
safe. The design lesson is to be deliberate: return new values instead of
mutating inputs when you don't own them, and prefer immutable types across module
boundaries so callers can't be surprised by aliasing.

---

### Q3. What is the `__eq__` / `__hash__` contract, and what breaks if you violate it?


**Deep dive.** The contract has two clauses. First, **objects that compare equal
must hash equal** (`a == b` ⇒ `hash(a) == hash(b)`); the reverse need not hold
(hash collisions are fine). Second, an object used as a dict key or set member
must be **hashable and effectively immutable for the fields that define
equality** — because the container places it in a bucket derived from its hash.
Violations fail *silently*, which is the danger. Override `__eq__` without
`__hash__` and Python sets `__hash__` to `None`, making instances unhashable — a
loud, early failure. Worse is defining both but inconsistently, or mutating a
key after insertion: the object lands in a bucket, its hash changes, and lookups
that "should" find it return a miss while the entry still occupies memory. That's
why value objects (see `Money` in `advanced_dunder.py`) are immutable and hash on
the same fields they compare on.

---

### Q4. Generators vs lists for a large or unbounded stream — what changes, and what is back-pressure?


**Deep dive.** A list computes and stores every element up front: O(n) memory and
all the work happens before you use the first item. A generator computes one
element per `next()` and suspends its frame in between: O(1) memory, work is
**lazy**, and it can represent an **infinite** sequence a list never could. The
architectural payoff for large streams is that memory stays flat regardless of
length, and you get natural **back-pressure** — because nothing is produced until
the consumer pulls, a slow consumer automatically throttles a fast producer
without any explicit buffer or queue management. The trade-offs are real: a
generator is single-pass (consume it twice and the second pass is empty), it
isn't indexable, and its laziness defers *when* exceptions surface, which can
confuse debugging. Choose a list when you need random access or multiple passes;
choose a generator when the data is large, streamed, or infinite.

---

### Q5. How does CPython free memory, and when is `weakref` the right tool rather than a normal reference?


**Deep dive.** CPython's primary mechanism is **reference counting**: every object
tracks how many references point at it, and it's freed *immediately* when the
count hits zero — deterministic, no pause. Refcounting has one blind spot:
**reference cycles** (A → B → A) keep each other's counts above zero forever, so a
**cyclic garbage collector** runs periodically to detect and reclaim unreachable
cycles. The consequence for design is that anything holding a strong reference
extends an object's lifetime — a cache, an observer list, a parent↔child
back-pattern can leak by keeping objects alive after their real owners are gone.
`weakref` is the right tool exactly there: it lets you *observe* or *cache* an
object **without** contributing to its refcount, so a `WeakValueDictionary`
entry disappears automatically when the last strong owner is dropped. Use it for
caches and back-references you don't want to own; use normal references
everywhere ownership is intended.

---

### Q6. When would you choose a `Protocol` over an ABC to define an interface?


**Deep dive.** Both express "this is the shape a collaborator must have," but they
differ in *how membership is decided*. An **ABC** is **nominal**: a type belongs
only if it explicitly subclasses (or is registered with) the ABC, and it can
provide shared implementation and enforce the contract at instantiation. A
**`Protocol`** is **structural**: any object with the right methods satisfies it,
with no inheritance and no import coupling — the implementer doesn't even need to
know the protocol exists. Prefer a `Protocol` when you're defining a dependency
your code *consumes*, especially across a boundary or when adapting third-party
types you can't subclass — it's how you get Dependency Inversion by shape and keep
tests trivial (pass a plain fake). Prefer an ABC when you own the hierarchy and
want to **share code** in the base or force subclasses to implement methods with a
runtime error. The senior answer names the axis — nominal-with-shared-code vs
structural-and-decoupled — instead of claiming one is universally better.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. `def add(x, items=[]): items.append(x); return items` — calling it three times with only `x` gives:**
- A) three separate one-element lists
- B) a growing shared list because the default is created once
- C) a `TypeError`
- D) an empty list each time

**2. Python's argument passing is best described as:**
- A) pass-by-value (arguments are copied)
- B) pass-by-reference (assigning the parameter changes the caller)
- C) call-by-object-reference (shared object; rebinding is local, mutation is visible)
- D) pass-by-name

**3. If you override `__eq__` on a class but not `__hash__`, instances are:**
- A) still hashable with the default hash
- B) unhashable (`__hash__` set to `None`)
- C) automatically frozen
- D) equal to everything

**4. The main advantage of a generator over a list for a 10-million-row stream is:**
- A) it can be indexed faster
- B) it uses ~constant memory and produces lazily (with back-pressure)
- C) it can be iterated many times
- D) it validates the data

**5. A reference *cycle* between two objects is reclaimed by:**
- A) reference counting alone
- B) the cyclic garbage collector
- C) `weakref`
- D) never — it always leaks

**6. `weakref.WeakValueDictionary` is the right choice when you want to:**
- A) keep cached values alive as long as the cache exists
- B) cache values without preventing them from being garbage-collected
- C) make dictionary access faster
- D) store unhashable keys

### Answer Key
1. **B** — the default list is created once at definition time and shared.
2. **C** — call-by-object-reference: mutation is visible, rebinding is local.
3. **B** — defining `__eq__` without `__hash__` sets `__hash__` to `None`.
4. **B** — constant memory, lazy production, natural back-pressure.
5. **B** — refcounting can't see cycles; the cyclic GC reclaims them.
6. **B** — weak values let entries vanish when the last strong owner is dropped.

---

## Part 3 — Gotchas Checklist

- **Mutable default arguments** (`=[]`, `={}`) are evaluated once and shared —
  default to `None` and build inside the function.
- **Aliasing** — `b = a` on a mutable object shares it; mutate through one and the
  other sees it. Copy or use immutables at boundaries.
- **`__eq__` without `__hash__`** makes instances unhashable; defining both
  inconsistently (or mutating a key) breaks dict/set lookups *silently*.
- **Generators are single-pass and not indexable** — consuming one twice yields
  nothing the second time; materialize to a list only when you truly need reuse.
- **Laziness defers errors** — an exception in a generator surfaces when it's
  pulled, not when it's created; account for this when debugging pipelines.
- **`functools.wraps`** — omit it and your decorator erases the wrapped
  function's name, docstring, and signature, breaking introspection and tooling.
- **Context managers that swallow exceptions** — returning truthy from `__exit__`
  suppresses errors; do it by accident and failures disappear.
- **Reference cycles + strong caches leak** — reach for `weakref` for caches and
  back-references you don't intend to own.
- **`__slots__` and descriptors are optimizations, not defaults** — apply them
  where the memory profile or the invariant justifies the lost flexibility.
- **Type hints don't enforce at runtime** — they power tooling and `Protocol`
  structural typing; validate real data at the boundary (Pydantic), not with hints.